# E10 Adult Census：折内目标编码与分组表现

> Issue #9 / Epic #6 / 路线图 E10

## 案例背景
Adult 收入预测含大量**高基数类别**（职业、教育、籍贯等）。One-Hot 会稀疏爆炸；**目标编码（Target Encoding）** 用目标均值压缩类别，但必须 **fold-safe**，否则验证集信息经全表统计泄漏进特征，离线 AUC 虚高。

## 你将学到
1. **泄漏演示**：全量 `y` 做 TE vs **OOF（Out-of-Fold）** 正确做法
2. **wanglaoshi API**：`FeatureEngineering.AdvancedFeatureEngineer.target_encode` — 便捷但**默认在传入的整张表上拟合**，课堂演示后必须强调上线要折内/嵌 Pipeline
3. 分组指标：按 `sex` 等切片看 Accuracy 差异（公平性分析入口）
4. `SubmissionGenerator`：`prediction_type='label'`

## 理论：目标编码与平滑
对类别 \(c\)，编码 \(\hat{\mu}_c = rac{n_c ar{y}_c + m ar{y}}{n_c + m}\)（\(m\) 为 smoothing）。  
**泄漏来源**：若用含验证行在内的全表估计 \(ar{y}_c\)，验证行的标签已通过统计量进入特征。

## 数据
- 本地：`data/adult.csv`
- 回退：OpenML `adult`（version 2）


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载 Adult


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/adult.csv')
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml('adult', version=2, as_frame=True, parser='auto')
    df = raw.frame.copy()
    print('OpenML', df.shape)

target_col = None
for c in ['income', 'class', 'target', '>50K']:
    if c in df.columns:
        target_col = c
        break
assert target_col, list(df.columns)

if QUICK and len(df) > 4000:
    df = df.sample(4000, random_state=RANDOM_STATE).reset_index(drop=True)
    print('QUICK sampled', df.shape)

print('target', target_col, df[target_col].value_counts())
df.head(3)


## 2. EDA


In [ ]:
from wanglaoshi import DataAnalyzer

eda_df = df.sample(min(len(df), 5000), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
display(analyzer.missing_value_analysis().head(15))
display(analyzer.basic_statistics().head(12))


## 3. 构造二值目标与特征列表


In [ ]:
from sklearn.preprocessing import LabelEncoder

work = df.copy()
y_raw = work[target_col].astype(str)
y = (y_raw.str.contains('>50K')).astype(int) if y_raw.str.contains('>50K').any() else LabelEncoder().fit_transform(y_raw)
work = work.drop(columns=[target_col])
cat_cols = work.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = work.select_dtypes(include=[np.number]).columns.tolist()
print('n_cat', len(cat_cols), 'n_num', len(num_cols))
print('sample cat', cat_cols[:6])


## 4. ❌ 错误示范：全量目标编码（泄漏）

下面故意用 **train+valid 全部行的 y** 估计类别均值，再划分。  
仅用于对比：你会看到「过于乐观」的近似指标倾向（本段用简单相关作泄漏直觉）。


In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    work, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
te_cols = cat_cols[:6]

def naive_full_te(frame, y_all, cols):
    out = frame.copy()
    for c in cols:
        means = y_all.groupby(frame[c].astype(str)).mean()
    return means

# 全表拼回去再编码（错误）
full = pd.concat([Xtr, Xte])
y_full = pd.concat([pd.Series(ytr, index=Xtr.index), pd.Series(yte, index=Xte.index)])
leak_tr, leak_te = Xtr.copy(), Xte.copy()
for c in te_cols:
    means = y_full.groupby(full[c].astype(str)).mean()
    gmean = y_full.mean()
    leak_tr[c + '_te'] = Xtr[c].astype(str).map(means).fillna(gmean)
    leak_te[c + '_te'] = Xte[c].astype(str).map(means).fillna(gmean)
feat_leak = num_cols + [c + '_te' for c in te_cols]
leak_tr_f = leak_tr[feat_leak].apply(pd.to_numeric, errors='coerce').fillna(0)
leak_te_f = leak_te[feat_leak].apply(pd.to_numeric, errors='coerce').fillna(0)
print('leakage feature shape', leak_tr_f.shape)


## 5. ✅ OOF 目标编码（折内拟合）

在训练集内 `KFold`：每折只用训练子集的 y 估计映射，对验证子集 **transform**。  
这是对 Kaggle/CV 友好的标准 TE 流程。


In [ ]:
from sklearn.model_selection import KFold

def oof_target_encode(X_train, y_train, cols, n_splits=5, seed=42):
    Xo = X_train.copy()
    y_s = pd.Series(y_train, index=X_train.index)
    gmean = y_s.mean()
    n_splits = 3 if QUICK else n_splits
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for c in cols:
        col_name = c + '_te_oof'
        Xo[col_name] = gmean
        for tr_i, va_i in kf.split(Xo):
            means = y_s.iloc[tr_i].groupby(X_train[c].astype(str).iloc[tr_i]).mean()
            Xo.iloc[va_i, Xo.columns.get_loc(col_name)] = (
                X_train[c].astype(str).iloc[va_i].map(means).fillna(gmean)
            )
    return Xo

def apply_te_from_train(train, valid, y_train, cols):
    tr = train.copy()
    va = valid.copy()
    y_s = pd.Series(y_train, index=train.index)
    gmean = y_s.mean()
    for c in cols:
        means = y_s.groupby(train[c].astype(str)).mean()
        tr[c + '_te'] = train[c].astype(str).map(means).fillna(gmean)
        va[c + '_te'] = valid[c].astype(str).map(means).fillna(gmean)
    return tr, va

Xtr_oof = oof_target_encode(Xtr, ytr, te_cols, seed=RANDOM_STATE)
Xtr_te, Xte_te = apply_te_from_train(Xtr, Xte, ytr, te_cols)
feat = num_cols + [c + '_te' for c in te_cols]
Xtr_f = Xtr_te[feat].apply(pd.to_numeric, errors='coerce').fillna(0)
Xte_f = Xte_te[feat].apply(pd.to_numeric, errors='coerce').fillna(0)
print('OOF cols sample', [c for c in Xtr_oof.columns if c.endswith('_te_oof')][:3])
print('holdout shape', Xtr_f.shape)


## 6. wanglaoshi `FE.target_encode` API 演示

`AdvancedFeatureEngineer.target_encode` 在**传入的 DataFrame 上直接 groupby 目标**。  
课堂用法：只在 **训练集子集** 上调用；并口头/WARN：**禁止**把 train+test 拼在一起调用。

下方在 `train` 切片上演示，再把映射应用到 `valid`（与 `apply_te_from_train` 等价思路）。


In [ ]:
from wanglaoshi import FeatureEngineering as FE

engineer = FE.AdvancedFeatureEngineer()
demo_col = te_cols[0]
train_slice = Xtr[[demo_col]].copy()
train_slice['__target__'] = ytr.values
print('⚠️ target_encode 必须在 fold-safe 流程中使用；下列仅在 train_slice 上演示 API')
encoded = engineer.target_encode(train_slice, demo_col, '__target__', smoothing=5.0)
enc_col = demo_col + '_target_encoded'
print(encoded[[demo_col, enc_col]].head(4))


## 7. 建模：缩放 + QuickBaseline（QUICK 跳 SVM）


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

scaler = StandardScaler()
Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr_f), columns=Xtr_f.columns, index=Xtr_f.index)
Xte_s = pd.DataFrame(scaler.transform(Xte_f), columns=Xte_f.columns, index=Xte_f.index)

if QUICK:
    models = {
        'Logistic Regression': LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE, n_jobs=-1),
        'Naive Bayes': GaussianNB(),
    }
    results = {}
    rows = []
    for name, model in models.items():
        model.fit(Xtr_s, ytr)
        pred_i = model.predict(Xte_s)
        acc = float((pred_i == np.asarray(yte)).mean())
        results[name] = {'model': model, 'predictions': pred_i}
        rows.append({'Model': name, 'Test Accuracy': acc})
    comparison = pd.DataFrame(rows).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
else:
    baseline = CT.QuickBaseline()
    results = baseline.run_all_models(Xtr_s, pd.Series(ytr), Xte_s, pd.Series(yte), task_type='classification')
    comparison = baseline.compare_models(results, top_n=5)

display(comparison)
best = comparison.iloc[0]['Model']
pred = results[best]['predictions']
print(classification_report(yte, pred, zero_division=0))


## 8. 泄漏 vs OOF 快速对照（同模型、不同特征）


In [ ]:
from sklearn.preprocessing import StandardScaler as SS

sc_l = SS()
Xtr_l = sc_l.fit_transform(leak_tr_f)
Xte_l = sc_l.transform(leak_te_f)
clf = LogisticRegression(max_iter=400, random_state=RANDOM_STATE)
clf.fit(Xtr_l, ytr)
acc_leak = (clf.predict(Xte_l) == np.asarray(yte)).mean()
clf2 = LogisticRegression(max_iter=400, random_state=RANDOM_STATE)
clf2.fit(Xtr_s, ytr)
acc_safe = (clf2.predict(Xte_s) == np.asarray(yte)).mean()
print('holdout acc (leaky TE)  ', round(acc_leak, 4))
print('holdout acc (train-map) ', round(acc_safe, 4))
print('若 leaky 明显更高，说明全量统计不可用于上线')


## 8.1 理论：smoothing 与稀有类别

目标编码 \(\hat{\mu}_c\) 向全局均值 \(\bar{y}\) 收缩，等价于贝叶斯视角下的小样本稳健化。  
`FE.target_encode(..., smoothing=5)` 越大 → 越接近全局均值，方差更小但偏差更大。  
高基数职业 `occupation` 等应配合 **OOF**，绝不可在全表上调用 `target_encode`。


In [ ]:
from wanglaoshi import FeatureEngineering as FE

eng = FE.AdvancedFeatureEngineer()
te_demo = Xtr[[te_cols[1]]].copy()
te_demo[te_cols[1]] = te_demo[te_cols[1]].astype(str)
fr = eng.frequency_encode(te_demo, te_cols[1])
print('frequency_encode 列', [c for c in fr.columns if 'frequency' in c][0])


## 9. 分组表现（性别等）


In [ ]:
group_col = None
for c in ['sex', 'Sex', 'gender', 'Gender']:
    if c in Xte.columns:
        group_col = c
        break
if group_col:
    tmp = pd.DataFrame({
        'y': np.asarray(yte),
        'pred': np.asarray(pred),
        'g': Xte[group_col].astype(str).to_numpy(),
    })
    print(tmp.groupby('g').apply(lambda d: (d['y'] == d['pred']).mean()))
else:
    print('无 sex/gender 列，跳过分组；可替换为 race 等')


## 9.1 公平性提示（入门级）

分组 Accuracy 不等时，不等于模型「歧视」，但提示应继续看 **组内 TPR/FPR、校准曲线**。  
Adult 历史数据含敏感属性，仅用于教学；上线前须做法务与伦理审查。


In [ ]:
if group_col:
    tmp2 = tmp.copy()
    tmp2['hit'] = tmp2['y'] == tmp2['pred']
    print('group size\n', tmp2.groupby('g').size())
    print('positive rate\n', tmp2.groupby('g')['y'].mean())


## 10. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/10_adult_census')
out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred)), 'income': np.asarray(yte)})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred,
    str(out / 'sample_submission.csv'),
    str(out / 'submission.csv'),
    prediction_type='label',
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head(3)


## 11. 检查清单
- [x] OOF / train-only TE 正确流程
- [x] 全量 TE 泄漏对照
- [x] `FE.target_encode` + fold-safe 警告
- [x] 分组表现入口
- [ ] 公平性：组间 TPR/FPR 差距
- [ ] 类别高基数：哈希 / 频率编码组合


## 附录 A：目标编码决策树

1. 类别基数 < 20 且样本足 → One-Hot 或 有序编码  
2. 基数中等、树模型 → 频率编码 / CatBoost 内置  
3. 基数高、线性/神经网络 → **OOF Target Encoding** + smoothing  
4. 生产 → 把 TE 写入 sklearn `Pipeline` + `ColumnTransformer`，折内 `fit`  

`FE.target_encode` 适合快速试验，但文档字符串级别就要记住：**传入 DataFrame 的范围 = 拟合范围**。课堂演示在 `train_slice` 调用；竞赛 CV 应用 `oof_target_encode` 同款逻辑或第三方 `category_encoders`。

## 附录 B：Adult 字段与泄漏源

- `fnlwgt`：抽样权重，通常不直接入模或仅作 sample_weight  
- `education` 与 `education-num`：勿同时保留冗余  
- `relationship` / `marital-status`：语义相关，注意共线  
- **最危险操作**：`pd.concat([train,test])` 后 `target_encode` —— 本 Notebook 第 4 节专门打脸此操作  


## 附录 C：OOF 与 hold-out 列对齐检查


In [ ]:
oof_cols = [c + '_te_oof' for c in te_cols if c + '_te_oof' in Xtr_oof.columns]
print('OOF 特征数', len(oof_cols))
print('hold-out 特征数', len(feat))
print('示例 OOF 列均值', Xtr_oof[oof_cols[:2]].mean().to_dict())


## 附录 D：读懂 wanglaoshi 文档中的 FE 警示

README / `book` 中多次强调：**目标编码统计量必须 fold-safe**。`AdvancedFeatureEngineer.target_encode` 不做折分，它是「单次 transform 工具」，不是 Kaggle Pipeline 替代品。正确做法：

1. 训练折估计映射  
2. 验证折 apply  
3. 测试折仅使用训练全集或各折模型集成后的映射（视策略而定）

与 `CompetitionTools.SubmissionGenerator` 联用时，预测列是离散收入标签（`<=50K` / `>50K` 的 0/1 表示），勿提交概率 CSV。


## 附录 E：类别基数一览


In [ ]:
cardinality = {c: Xtr[c].astype(str).nunique() for c in cat_cols}
top = sorted(cardinality.items(), key=lambda x: -x[1])[:10]
for c, n in top:
    print(c, n)


## 附录 F：从朴素贝叶斯到 TE 的演进路径

1. **LabelEncoder + 线性模型**：快但把类别当有序，误伤高基数列  
2. **One-Hot**：稀疏、维度爆炸  
3. **Frequency / Count**：无监督压缩，不泄漏标签但丢失目标信息  
4. **Target Encoding**：有监督、高收益、**必须 OOF**  
5. **CatBoost ordered boosting**：工业级折内编码，Adult 冲榜常用  

本 Notebook 路径 3→4→`FE.target_encode` API 演示，与 `QuickBaseline` 选型衔接。QUICK=True 时抽样 4000 行是为证据脚本 5 分钟内跑通；全量训练请设 QUICK=False 并改用 KFold TE Pipeline。


## 教师导读：Adult 与泄漏「应激训练」

**教学目标**：学生结束后能用一句话说明「为什么全表 target_encode 是错的」。

**演示顺序**：先跑 OOF（第 5 节）建立正确直觉，再跑全量 TE（第 4 节）制造对比；若 leaky acc 未显著高于 safe，解释抽样与 QUICK 子样本方差，强调趋势而非绝对差。

**FE API 警示板**：`target_encode` 写在幻灯片上：`fit scope = 传入 df 行集合`。让学生口述：「我对 train_fold 调 encode，对 valid_fold map，对 test 只用 train 全量 map。」

**分组公平性**：性别切片 Accuracy 差异可能是样本量与先验不同；引入 TPR parity 概念但不展开公式，防课时爆炸。

**作业建议**：（1）手写 3 折 OOF TE 伪代码；（2）用 `FE.frequency_encode` 替换一列 TE 比较；（3）阅读 `SubmissionGenerator` 文档确认禁止 probability 提交。

**课时衔接**：承接 E05 的「指标不能糊涂」——这里是「特征不能泄漏」。


## 附录 B：公平性与分组表现（简介）

Adult 常按性别/种族切分看 FPR/FNR 差异。正统课堂至少报告 **分组 Accuracy/F1**，避免只报整体。

这不等于做完公平性治理，但是「会拆分评估」的起点。


In [ ]:
print('After modeling: for g in gender groups: report f1/precision/recall separately.')
print('API map: FE.target_encode (fold-safe!), QuickBaseline, CrossValidator.stratified_cv')


## 附录 C：类别不平衡与阈值

>50K 比例约 25%。可尝试 class_weight 与阈值移动；主指标用 F1 或 Balanced Accuracy。


## 深度阅读与作业（Adult）

### 必读观点
1. 目标编码是强力特征，也是泄漏重灾区；必须 Pipeline/OOF。
2. sklearn `TargetEncoder` 内部 cross-fitting 的动机就是防自标签泄漏。
3. 除总分外，按性别等分组拆指标，是公平性讨论的最低要求。

### 书面作业
1. 复现「噪声类别 + 朴素 TE」虚高 AUC 的思想实验。
2. 对比 one-hot 与折内 TE 的 CV F1。
3. 列出本课用到的全部 `wanglaoshi` API。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：10_adult_census

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。
